# Pupil Function, PSF, and OTF


> Part of *PHYS 0241: Biomedical Imaging* (Prof. Michael E. Durst, Middlebury College). The [textbook page for this notebook](https://durstlab.middcreate.net/teaching/protected/textbook-biomedicalimaging/python/ch05/ch05-13.html) shows the rendered version together with the pre-rendered animations (course password required).
>
> **Start here:** no setup is needed for this notebook. In Google Colab, choose **File > Save a copy in Drive** to keep your changes.
>
> *Provided for academic use in PHYS 0241; please do not repost or redistribute ([terms of use](https://github.com/durstme/textbook-biomedicalimaging-notebooks/blob/main/LICENSE.md)).*

Fig 5.34 compares the coherent and incoherent cases for a **circular** aperture.
This notebook does the **1D slit** instead, because then every panel is a shape you
already know, and it adds the two things that figure leaves out: the **pupil function**
the chain starts from, and a **label on every arrow** so the relationship between the
quadrants is explicit.

Reading the diagram: the pupil is a rect, its Fourier transform is the coherent PSF
(a sinc), squaring gives the incoherent PSF (a sinc squared), and Fourier transforming
each of those gives the two transfer functions. The coherent one is the pupil back
again; the incoherent one is the pupil autocorrelated with itself, which is a triangle
of twice the width.

## Import libraries

NumPy and Matplotlib, and nothing else. That is a consequence of choosing the 1D
slit: the transform of a rect is a sinc, which NumPy already has. A circular
aperture would need `scipy.special.j1` for the Bessel function, as `ch05-06` does.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

## Define parameters

**Everything is measured in units of the coherent cutoff**, so `halfwidth1 = 1`
rather than a wavelength and a focal length. The shapes in this figure are the same
whatever lens you put in front of them; only the horizontal scale changes. `ch05-17`
and `ch05-19` put real micrometers on the same picture if you want them.

**Two different axes, and it matters which is which.** `nu1` is spatial frequency,
where the pupil and the transfer functions live. `x1` is position in the image plane,
where the spread functions live. The whole diagram is about moving between them.

**The four curves come from the pupil, and only from the pupil:**

- `pupil1` is a rect. This is the lens.
- `amplitude1` is its Fourier transform, a sinc. This is the coherent PSF $h$.
- `intensity1` is $\left\lvert h ightvert^2$, the **magnitude** squared. This is the incoherent PSF, and it is what a detector sees. For this real, even pupil $h$ is real so the magnitude does no work, but write it anyway: with defocus or aberration $h$ is complex and $h^2$ would be wrong.
- `otf_coh1` is the pupil again, because the coherent transfer function **is** the
  pupil function. Nothing is computed on that line, which is the point of it.
- `otf_inc1` is the pupil autocorrelated with itself, which for a rect is a triangle
  of twice the width. Hence the factor of two between the two cutoffs.

**Watch the sinc convention.** `np.sinc` is the *normalized* $\sin(\pi u)/(\pi u)$,
while this text writes $\mathrm{sinc}(u) = \sin(u)/u$. Here the normalized form is
actually the convenient one, because it puts the first zero at $x = 1/2a$ with no
factors of $\pi$ to carry around.

In [ ]:
COH = '#1f4fd8'      # coherent, blue as in ch05-06
INC = '#f08c1a'      # incoherent, orange as in ch05-06
PUP = '#444444'      # the pupil itself, neutral

# --- 1. Grids -------------------------------------------------------------
halfwidth1 = 1.0                       # pupil half-width, in units of the coherent cutoff
nu1 = np.linspace(-2.6, 2.6, 2001)     # spatial frequency axis (the pupil plane)
x1 = np.linspace(-6.0, 6.0, 2001)      # space axis (the image plane)

# --- 2. The pupil, and the two spread functions ---------------------------
pupil1 = (np.abs(nu1) <= halfwidth1).astype(float)

# h(x) = FT{P}: a slit of half-width a transforms to a sinc.
amplitude1 = np.sinc(2 * halfwidth1 * x1)          # numpy sinc is sin(pi u)/(pi u)
intensity1 = amplitude1 ** 2

# --- 3. The two transfer functions ---------------------------------------
# Coherent: the transfer function IS the pupil.
otf_coh1 = pupil1.copy()
# Incoherent: the autocorrelation of the pupil, normalized to 1 at zero frequency.
otf_inc1 = np.clip(1.0 - np.abs(nu1) / (2 * halfwidth1), 0.0, None)


def style(ax):
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.axhline(0, color='black', linewidth=1, alpha=0.5)
    ax.minorticks_on()
    ax.tick_params(which='both', direction="in", width=1)
    ax.tick_params(which='major', length=7)
    ax.tick_params(which='minor', length=3.5)
    ax.xaxis.set_ticks_position('both')
    ax.yaxis.set_ticks_position('both')


## Plot

Five panels arranged as the chain itself: **one pupil at the top, two branches
below it.**

The **top row** is the lens, spanning the full width because both branches start
from it. The **middle row** is the two spread functions, in space. The **bottom row**
is the two transfer functions, in frequency. Left column coherent, right column
incoherent, throughout.

**Every arrow carries a label**, which is the thing this figure exists to supply.
It is easy to collect these four curves as four separate facts; the labels say which
operation takes you from each one to the next, so the set becomes a single argument
rather than a list.

Three things are worth reading off it directly:

**The coherent transfer function is the pupil, unchanged.** Not proportional to it,
not shaped like it. It is the same function, which is why the left column closes back
on the top panel.

**The incoherent cutoff is exactly twice the coherent one.** That is not a separate
result to memorise: autocorrelating a function of width $w$ gives width $2w$, and the
incoherent transfer function is the pupil autocorrelated with itself. The factor of
two is a statement about widths, not about optics.

**Twice the cutoff is not twice as good.** The coherent transfer function is flat to
its edge, so everything it passes arrives at full strength. The incoherent one falls
from zero frequency onward, so mid-range detail is already weak long before the
cutoff. Compare the two bottom panels at any frequency inside the coherent cutoff and
the incoherent curve is always lower.

In [ ]:
fig = plt.figure(figsize=(13, 12))
gs = fig.add_gridspec(3, 2, hspace=1.05, wspace=0.58,
                      height_ratios=[0.72, 1.0, 1.0],
                      left=0.09, right=0.97, top=0.945, bottom=0.055)

# ---- Row 0: the pupil, spanning both columns ----
ax_p = fig.add_subplot(gs[0, :])
ax_p.plot(nu1, pupil1, color=PUP)
ax_p.fill_between(nu1, 0, pupil1, color=PUP, alpha=0.12)
ax_p.set_title(r"Pupil function $P(\nu)$:  the ideal lens is a rect",
               fontweight='bold')
ax_p.set_xlabel(r"spatial frequency $\nu$")
ax_p.set_ylabel(r"$P$")
ax_p.set_ylim(-0.15, 1.3)
ax_p.set_xlim(-2.6, 2.6)
ax_p.annotate("", xy=(halfwidth1, 1.12), xytext=(-halfwidth1, 1.12),
              arrowprops=dict(arrowstyle="<->", color=PUP, linewidth=1.8))
ax_p.text(0, 1.17, "aperture", ha='center', va='bottom', color=PUP, fontsize=12)
style(ax_p)

# ---- Row 1: the spread functions, in space ----
ax_h = fig.add_subplot(gs[1, 0])
ax_h.plot(x1, amplitude1, color=COH)
ax_h.set_title(r"Coherent PSF   $h(x)=\mathscr{F}\{P\}$", fontweight='bold', color=COH)
ax_h.set_xlabel("position $x$"); ax_h.set_ylabel("amplitude")
ax_h.set_ylim(-0.35, 1.15)
ax_h.text(0.03, 0.06, "sinc, so it goes negative", transform=ax_h.transAxes,
          fontsize=11, color=COH)
style(ax_h)

ax_h2 = fig.add_subplot(gs[1, 1])
ax_h2.plot(x1, intensity1, color=INC)
ax_h2.fill_between(x1, 0, intensity1, color=INC, alpha=0.15)
ax_h2.set_title(r"Incoherent PSF   $\left| h(x) \right|^2=|\mathscr{F}\{P\}|^2$",
                fontweight='bold', color=INC)
ax_h2.set_xlabel("position $x$"); ax_h2.set_ylabel("intensity")
ax_h2.set_ylim(-0.35, 1.15)
ax_h2.text(0.03, 0.06, "sinc$^2$, so it never does", transform=ax_h2.transAxes,
           fontsize=11, color=INC)
style(ax_h2)

# ---- Row 2: the transfer functions, in frequency ----
ax_c = fig.add_subplot(gs[2, 0])
ax_c.plot(nu1, otf_coh1, color=COH)
ax_c.fill_between(nu1, 0, otf_coh1, color=COH, alpha=0.12)
ax_c.set_title(r"Coherent OTF   $H(\nu)=P(\nu)$", fontweight='bold', color=COH)
ax_c.set_xlabel(r"spatial frequency $\nu$"); ax_c.set_ylabel("transfer factor")
ax_c.set_ylim(-0.15, 1.25); ax_c.set_xlim(-2.6, 2.6)
ax_c.scatter([halfwidth1], [0], color='black', s=45, zorder=5)
ax_c.annotate(r"cutoff $\frac{NA}{\lambda}$", xy=(halfwidth1, 0), xytext=(1.45, 0.55),
              arrowprops=dict(arrowstyle="->", color='black', linewidth=1.8), fontsize=15)
ax_c.text(0.03, 0.06, "flat, then a hard edge", transform=ax_c.transAxes,
          fontsize=11, color=COH)
style(ax_c)

ax_i = fig.add_subplot(gs[2, 1])
ax_i.plot(nu1, otf_inc1, color=INC)
ax_i.fill_between(nu1, 0, otf_inc1, color=INC, alpha=0.15)
ax_i.set_title(r"Incoherent OTF   $H(\nu)=P \star P$", fontweight='bold', color=INC)
ax_i.set_xlabel(r"spatial frequency $\nu$"); ax_i.set_ylabel("transfer factor")
ax_i.set_ylim(-0.15, 1.25); ax_i.set_xlim(-2.6, 2.6)
ax_i.scatter([2 * halfwidth1], [0], color='black', s=45, zorder=5)
ax_i.annotate(r"cutoff $\frac{2NA}{\lambda}$", xy=(2 * halfwidth1, 0), xytext=(1.15, 0.55),
              arrowprops=dict(arrowstyle="->", color='black', linewidth=1.8), fontsize=15)
ax_i.text(0.03, 0.06, "triangle, so it falls linearly", transform=ax_i.transAxes,
          fontsize=11, color=INC)
style(ax_i)

# ---- The labeled arrows: this is the point of the figure ----
# Positions are read back from the axes themselves rather than hard-coded, so the
# arrows stay in the gutters if the grid is ever retuned.
fig.canvas.draw()


def edge(ax_from, ax_to, label, vertical, pad=0.030):
    """Draw a labeled arrow in the gap between two axes."""
    a, b = ax_from.get_position(), ax_to.get_position()
    if vertical:
        x = (a.x0 + a.x1) / 2
        y0, y1 = a.y0 - pad, b.y1 + pad          # below one title, above the next
        p0, p1 = (x, y0), (x, y1)
    else:
        y = (a.y0 + a.y1) / 2
        x0, x1 = a.x1 + pad, b.x0 - pad          # across the column gutter
        p0, p1 = (x0, y), (x1, y)
    fig.patches.append(matplotlib.patches.FancyArrowPatch(
        p0, p1, transform=fig.transFigure, figure=fig,
        arrowstyle='-|>', mutation_scale=22, linewidth=2.0, color='black'))
    fig.text((p0[0] + p1[0]) / 2, (p0[1] + p1[1]) / 2, label,
             ha='center', va='center', fontsize=15, color='black',
             bbox=dict(boxstyle="round,pad=0.30", fc="white", ec='black', lw=1.2))


edge(ax_p,  ax_h,  r"$\mathscr{F}$",        vertical=True,  pad=0.038)
edge(ax_h,  ax_h2, r"$|\;\cdot\;|^{2}$",    vertical=False, pad=0.022)
edge(ax_h,  ax_c,  r"$\mathscr{F}$",        vertical=True,  pad=0.038)
edge(ax_h2, ax_i,  r"$\mathscr{F}$",        vertical=True,  pad=0.038)
edge(ax_c,  ax_i,  r"$\star$",              vertical=False, pad=0.022)
plt.show()
